Сделано с асгардархеей.

In [1]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00


In [25]:
from Bio import SeqIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

record = next(SeqIO.parse("gen.fna", "fasta"))
seq = str(record.seq[:50000]).upper()

nucleotides = ['A', 'C', 'G', 'T']
N = len(seq)

observed_counts = np.zeros((4, 4))
for i in range(N - 1):
    n1, n2 = seq[i], seq[i+1]
    if n1 in nucleotides and n2 in nucleotides:
        observed_counts[nucleotides.index(n1), nucleotides.index(n2)] += 1

single_counts = np.array([seq.count(n) for n in nucleotides])
p_i = single_counts / N

expected_counts = np.zeros((4, 4))
total_dinucleotides = np.sum(observed_counts)

for i in range(4):
    for j in range(4):
        expected_counts[i, j] = total_dinucleotides * p_i[i] * p_i[j]

chi_square_stat = np.sum((observed_counts - expected_counts)**2 / expected_counts)

df = 9
p_val = chi2.sf(chi_square_stat, df)

print(f"\nСтатистика Хи-квадрат: {chi_square_stat:.2f}")
print(f"p-value: {p_val:.2e}")

if p_val < 0.05:
    print("Результат: p < 0.05. Гипотеза независимости отвергается.")
    print("Вывод: В последовательности есть зависимости. Модель 1-го порядка (Марковская цепь) более адекватна.")
else:
    print("Результат: p >= 0.05. Гипотеза независимости принимается.")
    print("Вывод: Последовательность ведет себя как случайный шум (модель 0-го порядка).")

i_c, i_g = nucleotides.index('C'), nucleotides.index('G')
print(f"\nПример для CG:")
print(f"Наблюдалось: {observed_counts[i_c, i_g]}")
print(f"Ожидалось:   {expected_counts[i_c, i_g]:.1f}")


Статистика Хи-квадрат: 798.94
p-value: 3.60e-166
Результат: p < 0.05. Гипотеза независимости отвергается.
Вывод: В последовательности есть зависимости. Модель 1-го порядка (Марковская цепь) более адекватна.

Пример для CG:
Наблюдалось: 1003.0
Ожидалось:   1167.1
